# Lab Services Cleanup (lab.csv)

This notebook cleans the semi-structured **lab.csv** into a structured table with:
- `test_name`
- `test_category` (derived from each `Listing of ...` block)
- `category_section` (e.g., CHEMISTRY, SEROLOGY)
- 3 price columns (formatted with ₦)
- `test_key` (stable key)
- `sn_in_category` (original serial number within each category)


In [1]:
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 50)

In [2]:
df_PATH = "lab.csv"

# Load raw file as 5-column semi-structured text (no header)
df = pd.read_csv(df_PATH, header=None, dtype=str)
df = df.replace({np.nan: ""})  # keep blanks as empty strings

# Keep a copy of raw rows (we'll overwrite df later with the cleaned table)
df_raw = df.copy()

df_raw.shape

(927, 5)

In [3]:
# ---------------------------
# 2) Helpers
# ---------------------------
def _norm_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\ufeff", "")  # remove BOM if present
    s = re.sub(r"\s+", " ", s).strip()
    return s

def is_blank_row(row) -> bool:
    return all(_norm_text(x) == "" for x in row)

def is_category_row(row) -> bool:
    # Category rows look like: "Listing of RENAL ELECTROLYTE / BONE (CHEMISTRY)"
    c0 = _norm_text(row[0])
    return bool(re.match(r"(?i)^listing\s+of\b", c0))

def normalize_category(text: str) -> str:
    t = _norm_text(text)
    t = re.sub(r"(?i)^listing\s+of\s*", "", t).strip()
    return t

def extract_section(category: str) -> str:
    # Try to extract trailing "(CHEMISTRY)" etc
    m = re.search(r"\(([^)]+)\)\s*$", category or "")
    return _norm_text(m.group(1)) if m else ""

def is_header_row(row) -> bool:
    c0 = _norm_text(row[0]).upper()
    c1 = _norm_text(row[1]).lower()
    c2 = _norm_text(row[2]).lower()
    c3 = _norm_text(row[3]).lower()
    c4 = _norm_text(row[4]).lower()

    # Typical header: S/N | Name | Outsourced (B) | Walk in Patient (C) | Hospital Patient (D)
    return (
        c0 in {"S/N"}
        and c1 in {"name", "test", "test name"}
        and ("outsourced" in c2)
        and ("walk" in c3)
        and ("hospital" in c4)
    )

def parse_money(x):
    s = _norm_text(x)
    if s == "":
        return None
    # remove commas and any currency markers if they appear later
    s = s.replace(",", "")
    s = re.sub(r"[₦N\s]", "", s, flags=re.IGNORECASE)
    try:
        return float(s)
    except ValueError:
        return None

In [4]:
# ---------------------------
# 3) Parse the semi-structured file into a clean table
# ---------------------------
rows = df_raw.values.tolist()

current_category = ""
current_section = ""
in_table = False

records = []

for r in rows:
    # normalize to exactly 5 columns
    r = (list(r) + ["", "", "", "", ""])[:5]

    if is_blank_row(r):
        in_table = False
        continue

    if is_category_row(r):
        current_category = normalize_category(r[0])
        current_section = extract_section(current_category)
        in_table = False
        continue

    if is_header_row(r):
        in_table = True
        continue

    if not in_table:
        continue

    sn_df = _norm_text(r[0])
    name = _norm_text(r[1])

    # data rows should start with an integer S/N and have a name
    if not re.match(r"^\d+$", sn_df):
        continue
    if name == "":
        continue

    records.append(
        {
            "sn_in_category": int(sn_df),
            "test_name": name,
            "test_category": current_category,
            "category_section": current_section,  # e.g., CHEMISTRY / ENDOCRINOLOGY / SEROLOGY
            "price_outsourced": parse_money(r[2]),
            "price_walkin": parse_money(r[3]),
            "price_hospital": parse_money(r[4]),
        }
    )

# Overwrite df as the parsed (still-cleaning-needed) table
df = pd.DataFrame.from_records(records)

df.shape

(758, 7)

In [5]:
df.head(10)

,sn_in_category,test_name,test_category,category_section,price_outsourced,price_walkin,price_hospital
0,1,SODIUM,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,4000.0,4000.0,1000.0
1,2,POTASSIUM,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,4000.0,4000.0,1000.0
2,3,UREA,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,4000.0,4000.0,1000.0
3,4,PHOSPHATE(Serum),RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,4500.0,4500.0,1500.0
4,5,CREATININE,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,4000.0,4000.0,1000.0
5,6,CREATININE CLEARANCE(24 hour Urine),RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,10500.0,10500.0,5000.0
6,7,"ELECTROLYTES,UREA& CREATININE",RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,11500.0,11500.0,6000.0
7,8,ELECTROLYTES,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,7000.0,7000.0,4000.0
8,9,MAGNESIUM,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,10000.0,10000.0,4000.0
9,10,CALCIUM (Corrected),RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,7000.0,7000.0,2000.0


In [6]:
# ---------------------------
# 4) Cleaning / Standardization
# ---------------------------
# A) Standardize names (trim, collapse spaces)
df["test_name"] = df["test_name"].map(_norm_text)
df["test_category"] = df["test_category"].map(_norm_text)
df["category_section"] = df["category_section"].map(_norm_text)

# B) Convert prices to numeric
price_cols = ["price_outsourced", "price_walkin", "price_hospital"]
for c in price_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# C) Treat 0 as 'not priced / not available' (very common in the file)
KEEP_ZERO_AS_NA = True
if KEEP_ZERO_AS_NA:
    for c in price_cols:
        df.loc[df[c] == 0, c] = np.nan

# Keep a numeric copy for QA/stats (before we format the price columns to ₦ strings)
df_numeric = df.copy()

# D) Format the SAME 3 price columns with Naira (₦) — no extra *_naira columns
def fmt_naira(v):
    if pd.isna(v):
        return ""
    v = float(v)
    return f"₦{v:,.2f}"

for c in price_cols:
    df[c] = df[c].map(fmt_naira)

# E) Optional: a stable key you can use later for database lookups
def slugify(s: str) -> str:
    s = _norm_text(s).lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s

df["test_key"] = (df["test_category"].map(slugify) + "__" + df["test_name"].map(slugify))

# Final column order
cols = [
    "test_name",
    "test_category",
    "category_section",
    "price_outsourced",
    "price_walkin",
    "price_hospital",
    "test_key",
    "sn_in_category",
]
df = df[cols].reset_index(drop=True)

# Display without the left index column (index is not part of your data)
try:
    display(df.head(15).style.hide(axis="index"))
except Exception:
    display(df.head(15))

C:\Users\ACEMX\AppData\Local\Temp\ipykernel_23796\1387567481.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["test_name"] = df["test_name"].map(_norm_text)
C:\Users\ACEMX\AppData\Local\Temp\ipykernel_23796\1387567481.py:6: FutureWarni

test_name,test_category,category_section,price_outsourced,price_walkin,price_hospital,test_key,sn_in_category
SODIUM,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦4,000.00","₦4,000.00","₦1,000.00",renal_electrolyte_bone_chemistry__sodium,1
POTASSIUM,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦4,000.00","₦4,000.00","₦1,000.00",renal_electrolyte_bone_chemistry__potassium,2
UREA,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦4,000.00","₦4,000.00","₦1,000.00",renal_electrolyte_bone_chemistry__urea,3
PHOSPHATE(Serum),RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦4,500.00","₦4,500.00","₦1,500.00",renal_electrolyte_bone_chemistry__phosphate_serum,4
CREATININE,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦4,000.00","₦4,000.00","₦1,000.00",renal_electrolyte_bone_chemistry__creatinine,5
CREATININE CLEARANCE(24 hour Urine),RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦10,500.00","₦10,500.00","₦5,000.00",renal_electrolyte_bone_chemistry__creatinine_clearance_24_hour_urine,6
"ELECTROLYTES,UREA& CREATININE",RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦11,500.00","₦11,500.00","₦6,000.00",renal_electrolyte_bone_chemistry__electrolytes_urea_creatinine,7
ELECTROLYTES,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦7,000.00","₦7,000.00","₦4,000.00",renal_electrolyte_bone_chemistry__electrolytes,8
MAGNESIUM,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦10,000.00","₦10,000.00","₦4,000.00",renal_electrolyte_bone_chemistry__magnesium,9
CALCIUM (Corrected),RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,"₦7,000.00","₦7,000.00","₦2,000.00",renal_electrolyte_bone_chemistry__calcium_corrected,10


In [7]:
# ---------------------------
# 5) Quick QA checks
# ---------------------------
print("Total tests parsed:", len(df))

print("\nMissing price counts (numeric view):")
display(df_numeric[["price_outsourced","price_walkin","price_hospital"]].isna().sum())

# Duplicates (same name + category)
dups = df_numeric.duplicated(subset=["test_category", "test_name"], keep=False)
print("\nDuplicate rows (same test_name within the same test_category):", int(dups.sum()))
display(
    df_numeric.loc[dups, ["test_category","test_name","price_outsourced","price_walkin","price_hospital"]]
        .sort_values(["test_category","test_name"])
        .head(30)
)

Total tests parsed: 758

Missing price counts (numeric view):


price_outsourced    246
price_walkin        245
price_hospital      102
dtype: int64


Duplicate rows (same test_name within the same test_category): 4


,test_category,test_name,price_outsourced,price_walkin,price_hospital
255,HYPERTENTION / OTHER ENDOCRINE ( ENDOCRINOLOGY),DIRECT RENIN,70000.0,70000.0,15000.0
257,HYPERTENTION / OTHER ENDOCRINE ( ENDOCRINOLOGY),DIRECT RENIN,70000.0,70000.0,20000.0
43,LIVER / PANCRESE (CHEMISTRY),LDH,15500.0,15500.0,7000.0
47,LIVER / PANCRESE (CHEMISTRY),LDH,15500.0,15500.0,6500.0


In [8]:
# ---------------------------
# 6) Useful summaries (optional)
# ---------------------------
# Count tests per category
cat_counts = (
    df_numeric.groupby(["test_category", "category_section"], dropna=False)
        .size()
        .reset_index(name="n_tests")
        .sort_values("n_tests", ascending=False)
)
display(cat_counts.head(25))

# Basic numeric price stats
display(df_numeric[["price_outsourced","price_walkin","price_hospital"]].describe())

,test_category,category_section,n_tests
29,OTHERS (CHEMISTRY),CHEMISTRY,74
27,MICROBIOLOGY,,68
22,INFECTIVE(SEROLOGY),SEROLOGY,43
25,LIVER / PANCRESE (CHEMISTRY),CHEMISTRY,39
2,AUTO IMMUNE (SEROLOGY),SEROLOGY,37
15,HAEMATOLOGY ( GENERAL),GENERAL,37
16,HEPATITIS TEST (SEROLOGY),SEROLOGY,36
30,RENAL ELECTROLYTE / BONE (CHEMISTRY),CHEMISTRY,35
14,GENETICS,,34
28,ONCOLOGY,,31


,price_outsourced,price_walkin,price_hospital
count,512.000000,513.000000,656.000000
mean,45584.806641,45547.604288,33974.463415
std,71613.551221,71527.698784,50232.180521
min,344.000000,344.000000,1000.000000
25%,11500.000000,11500.000000,8000.000000
50%,25000.000000,25000.000000,15000.000000
75%,51442.000000,51442.000000,40000.000000
max,650000.000000,650000.000000,500000.000000


In [10]:
# ---------------------------
# 7) Build an "issues" table for review (recommended before export)
# ---------------------------

# Ensure test_key exists on BOTH frames (prevents KeyError)
if "test_key" not in df.columns:
    df["test_key"] = df["test_category"].map(slugify) + "__" + df["test_name"].map(slugify)

if "test_key" not in df_numeric.columns:
    df_numeric["test_key"] = df_numeric["test_category"].map(slugify) + "__" + df_numeric["test_name"].map(slugify)

# A) Missing price rows (based on numeric view)
missing_mask = df_numeric[["price_outsourced", "price_walkin", "price_hospital"]].isna().any(axis=1)

# Build missing-prices table from df (so you see ₦ formatting)
df_missing_prices = df.loc[missing_mask, [
    "sn_in_category", "test_name", "test_category", "category_section",
    "price_outsourced", "price_walkin", "price_hospital", "test_key"
]].copy()

df_missing_prices["issue"] = "Missing price (was 0/blank in source)"
df_missing_prices = df_missing_prices.sort_values(["test_category", "sn_in_category", "test_name"])

print("Rows with at least one missing price:", len(df_missing_prices))
display(df_missing_prices.head(30))

# B) Duplicate tests within same category (same test name) — detect using numeric, display using df
dup_mask = df_numeric.duplicated(subset=["test_category", "test_name"], keep=False)

df_duplicates = df.loc[dup_mask, [
    "sn_in_category", "test_name", "test_category", "category_section",
    "price_outsourced", "price_walkin", "price_hospital", "test_key"
]].copy()

df_duplicates["issue"] = "Duplicate test_name within same category (needs decision)"
df_duplicates = df_duplicates.sort_values(["test_category", "test_name"])

print("Duplicate rows:", len(df_duplicates))
display(df_duplicates)


Rows with at least one missing price: 270


,sn_in_category,test_name,test_category,category_section,price_outsourced,price_walkin,price_hospital,test_key,issue
667,4,INDIVIDUAL RAST ALLERGENS,ALLERGY,,,,"₦7,500.00",allergy__individual_rast_allergens,Missing price (was 0/blank in source)
672,9,PAUL FFP,ALLERGY,,,,"₦20,000.00",allergy__paul_ffp,Missing price (was 0/blank in source)
673,10,FITNESS PACKAGE FOR MEN,ALLERGY,,,,,allergy__fitness_package_for_men,Missing price (was 0/blank in source)
676,13,ALKALINE PHOSPHATASE ISOENZYMES(BONE FRACTION),ALLERGY,,,,"₦25,000.00",allergy__alkaline_phosphatase_isoenzymes_bone_fraction,Missing price (was 0/blank in source)
677,14,All Fish Panel:BCR/ABL T(9:22) + TEL/AML t(12:21) + MLL 11q rearrangement.,ALLERGY,,,,"₦157,400.00",allergy__all_fish_panel_bcr_abl_t_9_22_tel_aml_t_12_21_mll_11q_rearrangement,Missing price (was 0/blank in source)
679,16,DICLOFENAC ALLERGY TEST,ALLERGY,,,,"₦20,000.00",allergy__diclofenac_allergy_test,Missing price (was 0/blank in source)
681,18,MATERNAL SCREENING(DOUBLE MARKER TEST),ALLERGY,,,,,allergy__maternal_screening_double_marker_test,Missing price (was 0/blank in source)
687,24,FIBROID TEST,ALLERGY,,,,,allergy__fibroid_test,Missing price (was 0/blank in source)
688,25,ROTA VIRUS VACCINE,ALLERGY,,,,"₦10,000.00",allergy__rota_virus_vaccine,Missing price (was 0/blank in source)
689,26,PROTEIN ELECTROPHORESIS (SRL),ALLERGY,,,,,allergy__protein_electrophoresis_srl,Missing price (was 0/blank in source)


Duplicate rows: 4


,sn_in_category,test_name,test_category,category_section,price_outsourced,price_walkin,price_hospital,test_key,issue
255,11,DIRECT RENIN,HYPERTENTION / OTHER ENDOCRINE ( ENDOCRINOLOGY),ENDOCRINOLOGY,"₦70,000.00","₦70,000.00","₦15,000.00",hypertention_other_endocrine_endocrinology__direct_renin,Duplicate test_name within same category (needs decision)
257,13,DIRECT RENIN,HYPERTENTION / OTHER ENDOCRINE ( ENDOCRINOLOGY),ENDOCRINOLOGY,"₦70,000.00","₦70,000.00","₦20,000.00",hypertention_other_endocrine_endocrinology__direct_renin,Duplicate test_name within same category (needs decision)
43,9,LDH,LIVER / PANCRESE (CHEMISTRY),CHEMISTRY,"₦15,500.00","₦15,500.00","₦7,000.00",liver_pancrese_chemistry__ldh,Duplicate test_name within same category (needs decision)
47,13,LDH,LIVER / PANCRESE (CHEMISTRY),CHEMISTRY,"₦15,500.00","₦15,500.00","₦6,500.00",liver_pancrese_chemistry__ldh,Duplicate test_name within same category (needs decision)


In [11]:
# ---------------------------
# 8) EXPORTS
# ---------------------------

# FINAL_EXPORT (display) keeps ₦ formatting in the 3 price columns
FINAL_EXPORT_DISPLAY = df.copy()

# FINAL_EXPORT_NUMERIC keeps numeric values (best for DB / calculations)
FINAL_EXPORT_NUMERIC = df_numeric.copy()

# 1) Main clean exports
FINAL_EXPORT_DISPLAY.to_csv("lab_cleaned.csv", index=False, encoding="utf-8-sig")
FINAL_EXPORT_NUMERIC.to_csv("lab_cleaned_numeric.csv", index=False, encoding="utf-8-sig")
print("Saved: lab_cleaned.csv (₦ formatted)")
print("Saved: lab_cleaned_numeric.csv (numeric)")

# 2) Issues exports (for review)
df_missing_prices.to_csv("lab_issues_missing_prices.csv", index=False, encoding="utf-8-sig")
df_duplicates.to_csv("lab_issues_duplicates.csv", index=False, encoding="utf-8-sig")
print("Saved: lab_issues_missing_prices.csv")
print("Saved: lab_issues_duplicates.csv")

# Also export numeric issues (useful if you want to filter/sort in Excel later)
df_missing_prices_numeric = df_numeric.loc[
    missing_mask,
    ["sn_in_category", "test_name", "test_category", "category_section",
     "price_outsourced", "price_walkin", "price_hospital", "test_key"]
].copy()
df_missing_prices_numeric["issue"] = "Missing price (was 0/blank in source)"

df_duplicates_numeric = df_numeric.loc[
    dup_mask,
    ["sn_in_category", "test_name", "test_category", "category_section",
     "price_outsourced", "price_walkin", "price_hospital", "test_key"]
].copy()
df_duplicates_numeric["issue"] = "Duplicate test_name within same category (needs decision)"

df_missing_prices_numeric.to_csv("lab_issues_missing_prices_numeric.csv", index=False, encoding="utf-8-sig")
df_duplicates_numeric.to_csv("lab_issues_duplicates_numeric.csv", index=False, encoding="utf-8-sig")
print("Saved: lab_issues_missing_prices_numeric.csv")
print("Saved: lab_issues_duplicates_numeric.csv")

# 3) Excel bundle (nice for management + keeps everything together)
with pd.ExcelWriter("lab_cleaned_bundle.xlsx", engine="openpyxl") as writer:
    FINAL_EXPORT_DISPLAY.to_excel(writer, sheet_name="cleaned_lab_display", index=False)
    FINAL_EXPORT_NUMERIC.to_excel(writer, sheet_name="cleaned_lab_numeric", index=False)
    df_missing_prices.to_excel(writer, sheet_name="missing_prices_display", index=False)
    df_duplicates.to_excel(writer, sheet_name="duplicates_display", index=False)
    df_missing_prices_numeric.to_excel(writer, sheet_name="missing_prices_numeric", index=False)
    df_duplicates_numeric.to_excel(writer, sheet_name="duplicates_numeric", index=False)
    cat_counts.to_excel(writer, sheet_name="category_counts", index=False)

print("Saved: lab_cleaned_bundle.xlsx")


Saved: lab_cleaned.csv (₦ formatted)
Saved: lab_cleaned_numeric.csv (numeric)
Saved: lab_issues_missing_prices.csv
Saved: lab_issues_duplicates.csv
Saved: lab_issues_missing_prices_numeric.csv
Saved: lab_issues_duplicates_numeric.csv
Saved: lab_cleaned_bundle.xlsx
